# Verify Preprocessed Keypoint Data

This notebook loads preprocessed keypoint data and verifies:
1. Data structure and dimensions
2. Keypoint ordering matches XML sites
3. Scale matches MuJoCo model
4. Visualization with colored sites overlaid on model

In [1]:
import sys
from pathlib import Path

# Add project root to Python path FIRST to ensure our modules take priority
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"Added {project_root} to Python path")
elif sys.path.index(str(project_root)) != 0:
    # Move to front if it exists but isn't first
    sys.path.remove(str(project_root))
    sys.path.insert(0, str(project_root))
    print(f"Moved {project_root} to front of Python path")

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=True"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Use GPU 0

# JAX setup
import jax
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
# Note: jax_persistent_cache_enable_xla_caches may not be available in all JAX versions
try:
    jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")
except AttributeError:
    pass  # Skip if not available in this JAX version
jax.config.update("jax_default_matmul_precision", "high")

# Matplotlib setups
import matplotlib as mpl
mpl.rcParams.update({
    'font.size': 10,
    'axes.linewidth': 2,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.major.width': 2,
    'ytick.major.width': 2,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'pdf.fonttype': 42,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.facecolor': 'white',
    'pdf.use14corefonts': True,
    'svg.fonttype': 'none',
    'font.family': 'sans-serif',
    'font.serif': 'Arial',
})

Added /home/eabe/Research/MyRepos/3d_tracking_dataset to Python path


In [2]:
%load_ext autoreload
%autoreload 2

# Imports
import numpy as np
import jax.numpy as jnp
import mujoco
import matplotlib.pyplot as plt
import mediapy as media

from omegaconf import DictConfig, OmegaConf
from natsort import natsorted
from pathlib import Path
from tqdm.auto import tqdm

import utils.io_dict_to_hdf5 as ioh5
from utils.add_aligned_keypoint_sites import set_aligned_site_colors
from utils.stac_data_utils import print_bout_dict_structure
from utils.path_utils import (register_custom_resolvers, 
                              create_fresh_config_with_paths, 
                              convert_dict_to_path, 
                              convert_dict_to_string, 
                              load_config_and_override_paths)
register_custom_resolvers()
print("✓ Imports successful")

✓ Imports successful


## 1. Configuration

In [3]:
dataset = 'courtship' 
# base_dir = Path('/data2/users/eabe/datasets/Johnson_lab/free_walking')
base_dir = Path(f'/data2/users/eabe/datasets/Johnson_lab/{dataset}')
version_paths = sorted(list(base_dir.rglob('Predictions_3D*')))
# version_paths = sorted(list(base_dir.rglob('Data_analysis*')))
for n, version_temp in enumerate(version_paths):
    print(n,version_temp)
idx = 0
pre_version = version_paths[idx].name
print(f"Using version: {pre_version}")

0 /data2/users/eabe/datasets/Johnson_lab/courtship/2026_04_02_12_11_50/quality_viz/Predictions_3D_34327249_good_bouts.csv
1 /data2/users/eabe/datasets/Johnson_lab/courtship/2026_04_02_12_11_50/quality_viz/Predictions_3D_34327249_top0_f78250.png
2 /data2/users/eabe/datasets/Johnson_lab/courtship/2026_04_02_12_11_50/quality_viz/Predictions_3D_34327249_top1_f77750.png
3 /data2/users/eabe/datasets/Johnson_lab/courtship/2026_04_02_12_11_50/quality_viz/Predictions_3D_34327249_top2_f557750.png
4 /data2/users/eabe/datasets/Johnson_lab/courtship/Predictions_3D_20260123-093310
5 /data2/users/eabe/datasets/Johnson_lab/courtship/Predictions_3D_20260123-093310/videos/Predictions_3D_20260407-173851
6 /data2/users/eabe/datasets/Johnson_lab/courtship/Predictions_3D_20260123-093310/videos/Predictions_3D_20260407-175313
7 /data2/users/eabe/datasets/Johnson_lab/courtship/Predictions_3D_20260123-093310/videos/Predictions_3D_20260407-180417
8 /data2/users/eabe/datasets/Johnson_lab/courtship/Predictions_3D_

In [4]:
version = 'Data_analysis'
# base_dir = Path(f'/gscratch/portia/eabe/fly_neuromech/{version}')
base_dir = Path(f'/data2/users/eabe/datasets/Johnson_lab/{dataset}/{version}')
run_cfg_list = natsorted(list(Path(base_dir).rglob('combined_config.yaml')))
for n, run_cfg in enumerate(run_cfg_list):
    temp = OmegaConf.load(run_cfg)
    print(f'{n}:', temp.version, run_cfg)

# ###### Load and update config with specified paths template ###### 
cfg_num = 1

# NEW APPROACH: Load config and replace paths using workstation.yaml template
cfg = load_config_and_override_paths(
    config_path=run_cfg_list[cfg_num],
    new_paths_template="workstation",    # Use workstation.yaml for local paths
    config_dir=Path.cwd().parent / "configs",
    verbose=False
)

print(f'✅ Loaded experiment: {cfg_num}, {cfg.version}: {run_cfg_list[cfg_num]}')

# Convert string paths to Path objects and create directories
cfg.paths = convert_dict_to_path(cfg.paths)
print("✅ Successfully converted all paths to Path objects and created directories")

fig_dir = cfg.paths.fig_dir
media.set_show_save_dir(fig_dir)

# If combined data not already saved, run the combine_data script to create it
# cfg = create_fresh_config_with_paths(dataset="free_walking", paths_template="workstation", version=version, run_id="analysis", other_overrides=['anatomy=v1'])
# cfg.paths = convert_dict_to_path(cfg.paths)

0: Data_analysis /data2/users/eabe/datasets/Johnson_lab/courtship/Data_analysis/analysis/logs/combined_config.yaml
1: Data_analysis /data2/users/eabe/datasets/Johnson_lab/courtship/Data_analysis/analysis/v1/logs/combined_config.yaml
✅ Loaded experiment: 1, Data_analysis: /data2/users/eabe/datasets/Johnson_lab/courtship/Data_analysis/analysis/v1/logs/combined_config.yaml
✅ Successfully converted all paths to Path objects and created directories


In [5]:

# Preprocessed data file
use_interp = False
preprocessed_path = cfg.paths.base_dir.parent / f'{pre_version}/preprocessing/{cfg.preprocessing.bout_name}.h5'
stac_base = cfg.paths.save_dir / cfg.anatomy.name / cfg.dataset.get('concat', {}).get('output_file', 'ik_output_combined')
if stac_base.exists():
    if use_interp:
        stac_path = stac_base.parent / f"{stac_base.stem}_interpolated.h5"
    else:
        stac_path = stac_base
else:
    stac_path = cfg.paths.data_dir / f"postprocessing/{cfg.postprocessing.output_file}"
# Model paths

skeleton_path = project_root / 'data' / 'fly50.json'
flybody_path = Path(cfg.anatomy.mjcf_path)
# flybody_path = cfg.paths.body_model_dir /'fruitfly_v2.1/fruitfly_v2.1_muscles.xml' # Path(cfg.anatomy.mjcf_path) #
# flybody_path = cfg.paths.body_model_dir /'fruitfly_v1/fruitfly_v1_free.xml' # Path(cfg.anatomy.mjcf_path) #
floor_path = cfg.paths.body_model_dir / 'fruitfly_v2.1/floor.xml' # Path(cfg.anatomy.arena_path) #
print(f"Preprocessed data: {preprocessed_path}")
print(f"stac_path: {stac_path}")
print(f"Model: {flybody_path}")
print(f"Data exists: {preprocessed_path.exists()}")
print(f"STAC data exists: {stac_path.exists()}")
print(f"Model exists: {flybody_path.exists()}")


Preprocessed data: /data2/users/eabe/datasets/Johnson_lab/courtship/Predictions_3D_34327249_good_bouts.csv/preprocessing/preprocessed_bout_v1_courtship.h5
stac_path: /data2/users/eabe/datasets/Johnson_lab/courtship/Data_analysis/analysis/v1/ik_output_combined_v1_courtship.h5
Model: /home/eabe/Research/MyRepos/fruitfly_body_models/fruitfly_v1/fruitfly_v1_free.xml
Data exists: False
STAC data exists: True
Model exists: True


## 2. Load Preprocessed Data

In [6]:
# Load HDF5 data
print("Loading preprocessed data...")
data_dict = ioh5.load(preprocessed_path, enable_jax=True)
# data_dict = ioh5.load('/data2/users/eabe/datasets/Johnson_lab/free_walking/Predictions_3D_20260202-171900/preprocessed_bout_v1.h5', enable_jax=True)

print(f"\n✓ Loaded {len(data_dict)} bouts")
print(f"Bout keys: {list(data_dict.keys())[:5]}...")

# Examine first bout
bout_n = 0
bout_key = f'bout_{bout_n:03d}'
bout = data_dict[bout_key]

print(f"\nBout '{bout_key}' structure:")
for key, val in bout.items():
    if hasattr(val, 'shape'):
        print(f"  {key:20s}: shape {val.shape}, dtype {val.dtype}")
    elif isinstance(val, dict):
        print(f"  {key:20s}: dict with keys {list(val.keys())}")
    elif isinstance(val, list):
        print(f"  {key:20s}: list of {len(val)} items")
    else:
        print(f"  {key:20s}: {type(val).__name__}")

Loading preprocessed data...

✓ Loaded 66 bouts
Bout keys: ['bout_000', 'bout_001', 'bout_002', 'bout_003', 'bout_004']...

Bout 'bout_000' structure:
  alignment_info      : dict with keys ['exclude_indices', 'rotation', 'scales', 'translation']
  keypoints           : shape (233, 50, 3), dtype float32
  kp_names            : list of 50 items
  orig_keypoints      : shape (233, 50, 3), dtype float32
  skeleton_edges      : shape (44, 2), dtype int32


In [7]:
# Align MuJoCo model's T1R coxa to keypoint T1R coxa (FIXED)
from utils.add_aligned_keypoint_sites import (
    add_aligned_keypoint_sites_to_model, 
    get_aligned_site_indices,
    set_aligned_site_colors,
    add_aligned_mocap_bodies,
    get_aligned_mocap_indices
)

# Step 1: Find T1R_ThxCx index in keypoint data
# bout_n = 2
kp_data = data_dict[f'bout_{bout_n:03d}']['keypoints']  # Shape: (T, N, 3)
kp_node_names = data_dict[f'bout_{bout_n:03d}']['kp_names']

print(f"Keypoint data shape: {kp_data.shape}")
print(f"Available keypoints: {kp_node_names[:10]}...")

# Find T1R_ThxCx index
try:
    t1r_coxa_idx = kp_node_names.index('T1R_ThxCx')
    print(f"\n✓ Found T1R_ThxCx at index {t1r_coxa_idx}")
except ValueError:
    print("\n⚠ T1R_ThxCx not found in keypoint names!")
    print("Available keypoints with 'T1R':")
    for i, name in enumerate(kp_node_names):
        if 'T1R' in name:
            print(f"  [{i}] {name}")
    t1r_indices = [i for i, name in enumerate(kp_node_names) if 'T1R' in name]
    if t1r_indices:
        t1r_coxa_idx = t1r_indices[0]
        print(f"\nUsing {kp_node_names[t1r_coxa_idx]} at index {t1r_coxa_idx} as reference")
    else:
        raise ValueError("No T1R keypoints found!")

# Step 2: Load models and add mocap bodies
spec = mujoco.MjSpec().from_file(flybody_path.as_posix())

# Load floor and attach fly WITH floor offset
suffix = '_fly'
floor_offset = -0.125  # Floor z-position
floor_spec = mujoco.MjSpec().from_file(floor_path.as_posix())
spawn_frame = floor_spec.worldbody.add_frame(
    pos=[0, 0, floor_offset],
    quat=[1, 0, 0, 0],
)
spawn_body = spawn_frame.attach_body(spec.body("thorax"), "", suffix=suffix)

# Add mocap bodies for keypoint visualization
floor_spec = add_aligned_mocap_bodies(
    floor_spec, 
    kp_node_names, 
    color_coded=True, 
    prefix='aligned_'
)

# Compile model
mj_model = floor_spec.compile()
print(f"\n✓ Model compiled - Total mocap bodies: {mj_model.nmocap}")

# Get mocap indices for keypoint visualization
mocap_indices = get_aligned_mocap_indices(mj_model, kp_node_names, prefix='aligned_')
print(f"✓ Mapped {len(mocap_indices)} mocap bodies")

# Step 3: Get T1R coxa site ID from MuJoCo model
t1r_coxa_site_name = f'tracking[T1R_ThxCx]{suffix}'
try:
    t1r_coxa_site_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_SITE, t1r_coxa_site_name)
    print(f"✓ Found MuJoCo T1R coxa site: {t1r_coxa_site_name} (ID: {t1r_coxa_site_id})")
except:
    print(f"⚠ Could not find site: {t1r_coxa_site_name}")
    print(f"Available tracking sites:")
    for i in range(mj_model.nsite):
        site_name = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_SITE, i)
        if 'T1R' in site_name and 'tracking' in site_name:
            print(f"  {site_name}")
    raise ValueError("T1R coxa site not found in model")

# Initialize data
mj_data = mujoco.MjData(mj_model)
mujoco.mj_forward(mj_model, mj_data)

# Get T1R coxa position in model's reference pose
# This includes the floor offset already
ref_t1r_coxa_world = mj_data.site_xpos[t1r_coxa_site_id].copy()
print(f"\nT1R coxa in model reference (with floor offset): {ref_t1r_coxa_world}")
print(f"Floor offset: {floor_offset}")

# Get the root body position in reference pose
root_body_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, f'thorax{suffix}')
ref_root_pos = mj_data.xpos[root_body_id].copy()
print(f"Root body position in reference: {ref_root_pos}")

# Compute T1R coxa offset relative to root (in body frame)
t1r_coxa_offset_from_root = ref_t1r_coxa_world - ref_root_pos
print(f"T1R coxa offset from root: {t1r_coxa_offset_from_root}")

# Step 4: Render with aligned T1R coxa
scene_option = mujoco.MjvOption()
scene_option.sitegroup[:] = [1, 1, 1, 1, 1, 0]
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True

frames = []
num_frames =kp_data.shape[0]  # Limit for speed
print(f"\nRendering {num_frames} frames with T1R-coxa-aligned model...")

with mujoco.Renderer(mj_model, height=512, width=512) as renderer:
    for t in tqdm(range(num_frames)):
        # Get keypoint T1R coxa position for this frame
        kp_t1r_coxa_pos = kp_data[t, t1r_coxa_idx, :]
        
        # Set root position accounting for the body-to-coxa offset
        # root_pos + t1r_coxa_offset_from_root = kp_t1r_coxa_pos
        # Therefore: root_pos = kp_t1r_coxa_pos - t1r_coxa_offset_from_root
        root_pos = kp_t1r_coxa_pos - t1r_coxa_offset_from_root
        mj_data.qpos[:3] = root_pos
        
        # Forward kinematics to update model
        mujoco.mj_forward(mj_model, mj_data)
        
        # Verify alignment
        if t == 0:
            model_t1r_pos = mj_data.site_xpos[t1r_coxa_site_id]
            alignment_error = jnp.linalg.norm(model_t1r_pos - kp_t1r_coxa_pos)
            print(f"\nFrame 0 alignment check:")
            print(f"  Keypoint T1R coxa: {kp_t1r_coxa_pos}")
            print(f"  Model T1R coxa:    {model_t1r_pos}")
            print(f"  Alignment error:   {alignment_error:.9f}")
            
        # Update mocap body positions for keypoint visualization
        for i, mocap_id in mocap_indices.items():
            mj_data.mocap_pos[mocap_id] = kp_data[t, i, :]
            mj_data.mocap_quat[mocap_id] = [1, 0, 0, 0]
        
        # Render
        renderer.update_scene(mj_data, camera=f'track1{suffix}', scene_option=scene_option)
        pixels = renderer.render()
        frames.append(pixels)

# Display video
media.show_video(frames, fps=30)


Keypoint data shape: (233, 50, 3)
Available keypoints: ['Scutellum', 'WingL_base', 'WingR_base', 'Antenna_Base', 'EyeL', 'EyeR', 'WingL_V12', 'WingL_V13', 'WingR_V12', 'WingR_V13']...

✓ Found T1R_ThxCx at index 19
✓ Added 50 mocap bodies with colored sites

✓ Model compiled - Total mocap bodies: 50
✓ Mapped 50 mocap bodies
✓ Found MuJoCo T1R coxa site: tracking[T1R_ThxCx]_fly (ID: 24)

T1R coxa in model reference (with floor offset): [ 0.0317 -0.0209 -0.1522]
Floor offset: -0.125
Root body position in reference: [ 0.     0.    -0.125]
T1R coxa offset from root: [ 0.0317 -0.0209 -0.0272]

Rendering 233 frames with T1R-coxa-aligned model...


  0%|          | 0/233 [00:00<?, ?it/s]


Frame 0 alignment check:
  Keypoint T1R coxa: [0.6933519  0.46853596 0.17781937]
  Model T1R coxa:    [0.69335191 0.46853597 0.17781937]
  Alignment error:   0.000000000


This browser does not support the video tag.


# Post Stac Validation

In [7]:
# ── Load STAC output ──────────────────────────────────────────────────────────
# Works for both individual run files and combined batch files.
#
# Individual run (flat format):
#   .../Predictions_3D_XXXXX/stac/Fruitfly_ik_v1_free_walking.h5
# Combined batch (nested format):
#   .../Data_analysis/analysis/ik_output_combined_v1_free_walking.h5

# stac_file_path = '/data2/users/eabe/datasets/Johnson_lab/free_walking/Predictions_3D_20260202-171900/stac/Fruitfly_ik_v1_free_walking.h5'
# stac_file_path = stac_path  # path derived from cfg above
stac_file_path = '/data2/users/eabe/datasets/Johnson_lab/courtship/Data_analysis/analysis/v1/ik_output_combined_v1_courtship2.h5'
# stac_file_path = '/data2/users/eabe/datasets/Johnson_lab/free_walking/Predictions_3D_20260202-171900/postprocessing/ik_output_v1_free_walking.h5'  # path derived from cfg above
# stac_file_path2 = '/data2/users/eabe/datasets/Johnson_lab/free_walking/Predictions_3D_20260202-171900/ik_output_v1_free_walking.h5'
# stac_file_path = '/data2/users/eabe/datasets/Johnson_lab/free_walking/Predictions_3D_20260202-171900/postprocessing/ik_output_v2_muscles_free_walking.h5'  # path derived from cfg above
# stac_file_path2 = '/data2/users/eabe/datasets/Johnson_lab/free_walking/Predictions_3D_20260202-171900/ik_output_v2_muscles_free_walking.h5'


stac_data_raw = ioh5.load(stac_file_path, enable_jax=True)
# stac_data_raw2 = ioh5.load(stac_file_path2, enable_jax=True)
print(f"Loaded: {stac_file_path}")
print(f"Top-level keys: {list(stac_data_raw.keys())[:12]}")
print(f"N bouts: {len([key for key in stac_data_raw.keys() if key.startswith('bout_')])}")

Loaded: /data2/users/eabe/datasets/Johnson_lab/courtship/Data_analysis/analysis/v1/ik_output_combined_v1_courtship2.h5
Top-level keys: ['bout_000', 'bout_001', 'bout_002', 'bout_003', 'bout_004', 'bout_005', 'bout_006', 'bout_007', 'bout_008', 'bout_009', 'bout_010', 'bout_011']
N bouts: 25


In [15]:
stac_data_raw['info']['source_flies']

['fly0',
 'fly1',
 'fly0',
 'fly1',
 '',
 '',
 'fly0',
 'fly1',
 'fly0',
 'fly1',
 'fly0',
 'fly1',
 'fly0',
 'fly1',
 'fly0',
 'fly1',
 'fly0',
 'fly1',
 'fly0',
 'fly1',
 'fly0',
 'fly1',
 'fly1',
 'fly0',
 'fly1']

In [8]:
def _get_preprocessing_bout_lengths(config_str: str) -> tuple[list[int] | None, int | None]:
    """Parse the embedded config string and load preprocessing file bout lengths.

    Returns (bout_lengths, n_frames_per_clip) where:
      - bout_lengths: list of actual (unpadded) frame counts [T0, T1, ...] in bout order
      - n_frames_per_clip: padded clip length used during STAC
    Both are None if info cannot be recovered.
    """
    data_path = None
    n_frames_per_clip = None
    for line in config_str.split('\n'):
        stripped = line.strip()
        if stripped.startswith('data_path:') and 'preprocessing' in stripped:
            data_path = stripped.split('data_path:', 1)[1].strip()
        if stripped.startswith('n_frames_per_clip:'):
            n_frames_per_clip = int(stripped.split(':', 1)[1].strip())

    if data_path is None:
        return None, n_frames_per_clip

    pre_path = Path(data_path)
    if not pre_path.exists():
        print(f"  ⚠ Preprocessing file not found: {pre_path}")
        return None, n_frames_per_clip

    pre = ioh5.load(str(pre_path), enable_jax=False)
    bout_keys = sorted(k for k in pre if k.startswith('bout_') and isinstance(pre[k], dict))
    lengths = []
    for k in bout_keys:
        kp = pre[k].get('keypoints', pre[k].get('kp_data', None))
        if kp is not None:
            lengths.append(int(kp.shape[0]))
    print(f"  ✓ Loaded {len(lengths)} bouts from preprocessing: {pre_path.name}")
    print(f"    Actual lengths: {lengths}  (total={sum(lengths)})")
    print(f"    Padded clip length (n_frames_per_clip): {n_frames_per_clip}")
    return (lengths if lengths else None), n_frames_per_clip


def _normalize_stac_data(d: dict) -> dict:
    """Normalize individual-run or combined-batch STAC files to a common format.

    Individual run files have a flat structure with all clips concatenated and
    padded to n_frames_per_clip:
        qpos shape: (n_clips * n_frames_per_clip, nq)

    Combined batch files (output of combine_data) have a nested structure:
        info -> {clip_lengths, names_qpos, names_xpos, ...}
        bout_000, bout_001, ... -> {qpos, xpos, xquat, ...}

    This converts individual files to the nested format so the rest of the
    notebook works unchanged for both input types.
    """
    _INDIVIDUAL_KEYS = {'qpos', 'xpos', 'xquat', 'names_qpos', 'names_xpos'}
    is_individual = _INDIVIDUAL_KEYS.issubset(d.keys()) and 'info' not in d

    if not is_individual:
        return d  # already in combined format

    T_total = int(d['qpos'].shape[0])
    print(f"Individual run file detected — {T_total} total frames")

    # Recover actual bout lengths and padded clip size from the embedded config
    bout_lengths, n_frames_per_clip = None, None
    if 'config' in d:
        bout_lengths, n_frames_per_clip = _get_preprocessing_bout_lengths(str(d['config']))

    # Validate: T_total should equal n_clips * n_frames_per_clip
    if n_frames_per_clip is not None and T_total % n_frames_per_clip == 0:
        n_clips = T_total // n_frames_per_clip
        print(f"  → {n_clips} padded clip(s) of {n_frames_per_clip} frames each")
    else:
        n_clips = 1
        n_frames_per_clip = T_total

    # Validate bout_lengths against n_clips
    if bout_lengths is not None and len(bout_lengths) != n_clips:
        print(f"  ⚠ Bout count mismatch: preprocessing has {len(bout_lengths)} bouts "
              f"but {n_clips} padded clips found. Falling back to full clip lengths.")
        bout_lengths = None

    # Fall back: use the full padded clip length for each clip
    if bout_lengths is None:
        bout_lengths = [n_frames_per_clip] * n_clips
        print(f"  ⚠ Using padded lengths — output may include zero-padded tail frames")

    n_bouts = len(bout_lengths)
    print(f"  → Extracting {n_bouts} bout(s) with actual lengths: {bout_lengths}")

    # Keys that are per-frame arrays (axis 0 = time)
    array_keys = ['qpos', 'xpos', 'xquat', 'marker_sites', 'kp_data', 'qvel']

    # Each bout i lives at [i*n_frames_per_clip : i*n_frames_per_clip + actual_length_i]
    normalized = {
        'info': {
            'clip_lengths': jnp.array(bout_lengths),
            'names_qpos': list(d['names_qpos']),
            'names_xpos': list(d['names_xpos']),
        },
    }
    shared_keys = ['offsets', 'kp_names']
    for i, actual_len in enumerate(bout_lengths):
        start = i * n_frames_per_clip
        end   = start + actual_len
        bout_dict = {}
        for k in array_keys:
            if k in d and hasattr(d[k], 'shape') and d[k].shape[0] == T_total:
                bout_dict[k] = d[k][start:end]
        for k in shared_keys:
            if k in d:
                bout_dict[k] = d[k]
        normalized[f'bout_{i:03d}'] = bout_dict

    return normalized


# stac_data = stac_data_raw
stac_data = _normalize_stac_data(stac_data_raw)
# stac_data2 = _normalize_stac_data(stac_data_raw2)
print_bout_dict_structure(stac_data, max_depth=3)


BOUT DICTIONARY STRUCTURE
Total keys: 26 (info + 25 bouts)

├── bout_000: <dict: 9 keys>
│   ├── geometric_angles: <dict: 6 keys>
│   │   ├── T1L: <dict: 6 keys>
│   │   │   ├── T1L_FeTi_flex: <jax array: shape=(308,), dtype=float32>
│   │   │   ├── T1L_ThxCx_abduct: <jax array: shape=(308,), dtype=float32>
│   │   │   ├── T1L_ThxCx_flex: <jax array: shape=(308,), dtype=float32>
│   │   │   ├── T1L_ThxCx_rot: <jax array: shape=(308,), dtype=float32>
│   │   │   ├── T1L_Tro_flex: <jax array: shape=(308,), dtype=float32>
│   │   │   └── T1L_Tro_rot: <jax array: shape=(308,), dtype=float32>
│   │   ├── T1R: <dict: 6 keys>
│   │   │   ├── T1R_FeTi_flex: <jax array: shape=(308,), dtype=float32>
│   │   │   ├── T1R_ThxCx_abduct: <jax array: shape=(308,), dtype=float32>
│   │   │   ├── T1R_ThxCx_flex: <jax array: shape=(308,), dtype=float32>
│   │   │   ├── T1R_ThxCx_rot: <jax array: shape=(308,), dtype=float32>
│   │   │   ├── T1R_Tro_flex: <jax array: shape=(308,), dtype=float32>
│   │   │ 

In [9]:
def make_videos(
    mj_model,
    mj_data,
    qposes_rollout,
    scene_option,
    camera="track1",
    height=512,
    width=512,
    kp_data=None,
    mocap_indices=None,
    kp_anchor_idx=None,
    model_anchor_site_id=None,
):
    """
    Make a video of the rollout and reference superimposed.
    
    Args:
        kp_data: Optional (T, N_kp, 3) keypoint positions to overlay as mocap bodies.
        mocap_indices: Optional dict mapping keypoint index to mocap ID (from get_aligned_mocap_indices).
        kp_anchor_idx: Index of anchor keypoint (e.g. Scutellum) in kp_data for alignment.
        model_anchor_site_id: MuJoCo site ID of the corresponding anchor site on the body model.
    """
    frames = []
    with mujoco.Renderer(mj_model, height=height, width=width) as renderer:
        for t in tqdm(range(len(qposes_rollout))):
            mj_data.qpos = qposes_rollout[t]
            mujoco.mj_forward(mj_model, mj_data)
            
            # Update mocap body positions for keypoint overlay
            if kp_data is not None and mocap_indices is not None:
                # Compute offset to align kp anchor to model anchor
                offset = np.zeros(3)
                if kp_anchor_idx is not None and model_anchor_site_id is not None:
                    model_anchor_pos = mj_data.site_xpos[model_anchor_site_id]
                    kp_anchor_pos = kp_data[t, kp_anchor_idx, :]
                    offset = model_anchor_pos - kp_anchor_pos
                
                for kp_idx, mocap_id in mocap_indices.items():
                    mj_data.mocap_pos[mocap_id] = kp_data[t, kp_idx, :] + offset
                    mj_data.mocap_quat[mocap_id] = [1, 0, 0, 0]
            
            renderer.update_scene(
                mj_data, camera=f"{camera}", scene_option=scene_option
            )
            renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = False
            pixels = renderer.render()
            frames.append(pixels)
    return frames

In [20]:
from utils.add_aligned_keypoint_sites import add_aligned_mocap_bodies, get_aligned_mocap_indices

clip_lengths = jnp.asarray(stac_data['info']['clip_lengths'])
bout = 0# np.argmax(clip_lengths)
spec = mujoco.MjSpec().from_file(flybody_path.as_posix())
floor_spec = mujoco.MjSpec().from_file(floor_path.as_posix())
spawn_frame = floor_spec.worldbody.add_frame(
                pos=[0,0,0],
                quat=[1,0,0,0],
            )
spawn_body = spawn_frame.attach_body(spec.body("thorax"), "", suffix='_fly')
if not use_interp:
    end_eff_idxs = jnp.asarray([n for n, name in enumerate(stac_data['info']['names_xpos']) if 'claw' in name])
    floor_z_all = []
    for bout_temp in range(len(clip_lengths)):
        end_z = stac_data[f'bout_{bout_temp:03d}']['xpos'][:, end_eff_idxs, 2]
        floor_z = np.mean(np.quantile(end_z, [0.05], axis=0))
        floor_z_all.append(floor_z)
    floor_z_all = np.array(floor_z_all)
    floor_spec.geom('floor').pos[2] = np.mean(floor_z_all)

# Add mocap bodies for keypoint overlay
kp_names = stac_data['info'].get('kp_names', stac_data[f'bout_{bout:03d}'].get('kp_names', None))
if kp_names is not None:
    floor_spec = add_aligned_mocap_bodies(floor_spec, kp_names, color_coded=True, prefix='aligned_')

mj_model = floor_spec.compile()
mj_data = mujoco.MjData(mj_model)

# Get mocap indices for keypoint visualization
mocap_indices = None
bout_kp_data = None
kp_anchor_idx = None
model_anchor_site_id = None

if kp_names is not None:
    mocap_indices = get_aligned_mocap_indices(mj_model, kp_names, prefix='aligned_')
    bout_kp_data = stac_data[f'bout_{bout:03d}'].get('kp_data', None)
    if bout_kp_data is not None:
        bout_kp_data = np.asarray(bout_kp_data)
        # Reshape from (T, n_kp*3) to (T, n_kp, 3) if stored flat
        if bout_kp_data.ndim == 2 and bout_kp_data.shape[1] == len(kp_names) * 3:
            bout_kp_data = bout_kp_data.reshape(bout_kp_data.shape[0], len(kp_names), 3)
        print(f"Keypoint overlay enabled: {len(kp_names)} keypoints, kp_data shape: {bout_kp_data.shape}")

        # Find Scutellum anchor for alignment
        if 'Scutellum' in kp_names:
            kp_anchor_idx = kp_names.index('Scutellum')
            try:
                model_anchor_site_id = mujoco.mj_name2id(
                    mj_model, mujoco.mjtObj.mjOBJ_SITE, 'tracking[Scutellum]_fly')
                print(f"Scutellum alignment: kp index={kp_anchor_idx}, site id={model_anchor_site_id}")
            except:
                print("Warning: tracking[Scutellum]_fly site not found in model, no alignment")
                model_anchor_site_id = None
    else:
        print("No kp_data found in bout - keypoint overlay disabled")
        mocap_indices = None

t = 0
frames=[]
# qpos_traj =  jnp.concatenate([stac_data[f'bout_{bout:03d}']['qpos'].copy() for bout in [40, 109, 113]],axis=0)
qpos_traj =  stac_data[f'bout_{bout:03d}']['qpos'].copy()

# Trim kp_data to match qpos length
if bout_kp_data is not None:
    bout_kp_data = bout_kp_data[:len(qpos_traj)]

# Set up rendering
height, width = 448, 1936//3
camera = mj_model.camera(0).name
scene_option = mujoco.MjvOption()
scene_option.geomgroup[:] = [1, 1, 1, 0, 0, 0]
scene_option.sitegroup[:] = [1, 1, 1, 1, 1, 0]
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True

# Render from multiple camera angles
all_frames = []
for cam in np.asarray([1,2]):
    camera = mj_model.camera(cam).name
    frames = make_videos(mj_model, mujoco.MjData(mj_model), qpos_traj, scene_option,
                         camera=camera, height=height, width=width,
                         kp_data=bout_kp_data, mocap_indices=mocap_indices,
                         kp_anchor_idx=kp_anchor_idx, model_anchor_site_id=model_anchor_site_id)
    all_frames.append(frames)
all_frames = np.concatenate(all_frames, axis=2)
media.show_video(all_frames, fps=60, title=f'clip_concat')

✓ Added 50 mocap bodies with colored sites
Keypoint overlay enabled: 50 keypoints, kp_data shape: (308, 50, 3)
Scutellum alignment: kp index=0, site id=2


  0%|          | 0/308 [00:00<?, ?it/s]

  0%|          | 0/308 [00:00<?, ?it/s]

clip_concat This browser does not support the video tag.


In [ ]:
clip_lengths = jnp.asarray(stac_data['info']['clip_lengths'])
bout = 47
nq = int(stac_data[f'bout_{bout:03d}']['qpos'].shape[1])  # DOFs per single fly

# ── Build combined model: two flies in one scene ──────────────────────────────
spec1 = mujoco.MjSpec().from_file(flybody_path.as_posix())
spec2 = mujoco.MjSpec().from_file(flybody_path.as_posix())
floor_spec = mujoco.MjSpec().from_file(floor_path.as_posix())

floor_spec.worldbody.add_frame(pos=[0,0,0], quat=[1,0,0,0]).attach_body(
    spec1.body("thorax"), "", suffix='_fly')
floor_spec.worldbody.add_frame(pos=[0,0,0], quat=[1,0,0,0]).attach_body(
    spec2.body("thorax"), "", suffix='_fly2')

# Floor height from fly1 claw positions
end_eff_idxs = jnp.asarray([n for n, name in enumerate(stac_data['info']['names_xpos']) if 'claw' in name])
floor_z_all = []
for bout_temp in range(len(clip_lengths)):
    end_z = stac_data[f'bout_{bout_temp:03d}']['xpos'][:, end_eff_idxs, 2]
    floor_z_all.append(np.mean(np.quantile(np.array(end_z), [0.05], axis=0)))
floor_spec.geom('floor').pos[2] = np.mean(floor_z_all)

mj_model = floor_spec.compile()
mj_data  = mujoco.MjData(mj_model)

# ── Ghost effect: tint fly2 geoms blue + semi-transparent ─────────────────────
for i in range(mj_model.ngeom):
    gname = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_GEOM, i)
    if gname and '_fly2' in gname:
        r, g, b, a = mj_model.geom_rgba[i]
        # Shift colour toward cyan/blue, reduce alpha to ~35 %
        mj_model.geom_rgba[i] = [r * 0.2, g * 0.7, min(b * 1.5 + 0.3, 1.0), 1.0]

# ── Trajectories ──────────────────────────────────────────────────────────────
qpos_traj  = np.array(stac_data[f'bout_{bout:03d}']['qpos'].copy()[:1000])
qpos_traj2 = np.array(stac_data2[f'bout_{bout:03d}']['qpos'].copy()[:1000])
T_render = min(len(qpos_traj), len(qpos_traj2))

# ── Render ─────────────────────────────────────────────────────────────────────
height, width = 448, 1936 // 3
scene_option = mujoco.MjvOption()
scene_option.geomgroup[:] = [1, 1, 1, 0, 0, 0]
scene_option.sitegroup[:] = [1, 1, 1, 1, 1, 0]
scene_option.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = False   # enables alpha blending
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = False

all_frames = []
for cam_idx in [1, 2]:
    camera = mj_model.camera(cam_idx).name
    _mj_data = mujoco.MjData(mj_model)
    frames = []
    with mujoco.Renderer(mj_model, height=height, width=width) as renderer:
        for t in tqdm(range(T_render), desc=f'cam {cam_idx}'):
            _mj_data.qpos[:nq]      = qpos_traj[t]   # fly1 — solid
            _mj_data.qpos[nq:2*nq]  = qpos_traj2[t]  # fly2 — ghost (blue)
            mujoco.mj_forward(mj_model, _mj_data)
            renderer.update_scene(_mj_data, camera=camera, scene_option=scene_option)
            renderer.scene.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = False
            frames.append(renderer.render())
    all_frames.append(frames)

all_frames = np.concatenate(all_frames, axis=2)
media.show_video(all_frames, fps=60, title='ghost_fly_comparison  (solid=fly1  blue-ghost=fly2)')

In [ ]:
[mj_model.camera(cam).name for cam in range(mj_model.ncam)]

In [ ]:
# stac_data[f'bout_{bout:03d}']['qpos'][0,:3]
stac_data['info'].keys()
# stac_data[f'bout_{bout:03d}'].keys()


In [ ]:
# stac_data[f'bout_{bout:03d}']['xpos_egocentric'].shape
coxa_idx = np.asarray([idx for idx, name in enumerate(stac_data['info']['names_qpos']) if 'coxa' in name])
femur_idx = np.asarray([idx for idx, name in enumerate(stac_data['info']['names_qpos']) if 'femur' in name])
[stac_data['info']['names_qpos'][n] for n in coxa_idx]

In [ ]:
n_bouts = len(jnp.asarray(stac_data['info']['clip_lengths']))
bout = min(47, n_bouts - 1)  # clamp to available range
print(f"Plotting bout {bout} / {n_bouts} bouts")

names_qpos = stac_data['info']['names_qpos']

# Group coxa indices by leg — each leg may have 1 or 3 DOFs depending on anatomy
leg_patterns = [
    ('T1L', ['T1_left',  'T1L']),
    ('T1R', ['T1_right', 'T1R']),
    ('T2L', ['T2_left',  'T2L']),
    ('T2R', ['T2_right', 'T2R']),
    ('T3L', ['T3_left',  'T3L']),
    ('T3R', ['T3_right', 'T3R']),
]
coxa_groups = {}
for label, patterns in leg_patterns:
    idxs = [i for i, n in enumerate(names_qpos)
            if 'coxa' in n and any(p in n for p in patterns)]
    coxa_groups[label] = idxs
    print(f"  {label}: {[names_qpos[i] for i in idxs]}")

fig, axs = plt.subplots(3, 2, figsize=(10, 6), sharex=True, sharey=True)
axs = axs.flatten()
for ax_i, (label, idxs) in enumerate(coxa_groups.items()):
    ax = axs[ax_i]
    q1 = np.rad2deg(np.array(stac_data[f'bout_{bout:03d}']['qpos'])[:, idxs])
    # q2 = np.rad2deg(np.array(stac_data2[f'bout_{bout:03d}']['qpos'])[:, idxs])
    dof_labels = [names_qpos[i].replace('coxa_', '').split('_')[0] for i in idxs]
    for j, dlabel in enumerate(dof_labels):
        l, = ax.plot(q1[:, j], label=dlabel)
        # ax.plot(q2[:, j], '--', color=l.get_color(), alpha=0.6)
    ax.set_title(label)
    if len(dof_labels) > 1:
        ax.legend(fontsize=7, loc='upper right')

axs[4].set_xlabel('Frame')
axs[5].set_xlabel('Frame')
fig.text(0.04, 0.5, 'Coxa angle (°)', va='center', rotation='vertical')
fig.suptitle(f'Bout {bout} coxa joints  (solid = stac_data, dashed = stac_data2)', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
n_bouts = len(jnp.asarray(stac_data['info']['clip_lengths']))
bout = min(109, n_bouts - 1)
names_qpos = stac_data['info']['names_qpos']

# Also plot femur joints for comparison
joint_type = 'femur'  # change to 'coxa', 'tibia', etc.
leg_patterns = [
    ('T1L', ['T1_left',  'T1L']),
    ('T1R', ['T1_right', 'T1R']),
    ('T2L', ['T2_left',  'T2L']),
    ('T2R', ['T2_right', 'T2R']),
    ('T3L', ['T3_left',  'T3L']),
    ('T3R', ['T3_right', 'T3R']),
]
joint_groups = {}
for label, patterns in leg_patterns:
    idxs = [i for i, n in enumerate(names_qpos)
            if joint_type in n and any(p in n for p in patterns)]
    joint_groups[label] = idxs

fig, axs = plt.subplots(3, 2, figsize=(10, 6), sharex=True)
axs = axs.flatten()
for ax_i, (label, idxs) in enumerate(joint_groups.items()):
    ax = axs[ax_i]
    if not idxs:
        ax.set_title(f'{label} (no {joint_type})')
        continue
    q1 = np.rad2deg(np.array(stac_data[f'bout_{bout:03d}']['qpos'])[:, idxs])
    # q2 = np.rad2deg(np.array(stac_data2[f'bout_{bout:03d}']['qpos'])[:, idxs])
    dof_labels = [names_qpos[i] for i in idxs]
    for j, dlabel in enumerate(dof_labels):
        l, = ax.plot(q1[:, j], label=dlabel.split('_')[0])
        # ax.plot(q2[:, j], '--', color=l.get_color(), alpha=0.6)
    ax.set_title(label)

axs[4].set_xlabel('Frame')
axs[5].set_xlabel('Frame')
fig.text(0.04, 0.5, f'{joint_type.capitalize()} angle (°)', va='center', rotation='vertical')
fig.suptitle(f'Bout {bout} {joint_type} joints  (solid = stac_data, dashed = stac_data2)', fontsize=10)
plt.tight_layout()
plt.show()
